# Survival Analysis

## Python 

There are (at least) two packages performing survival analysis in python: `lifelines` and `sksurv`. You can use either of them to perform tasks below.

## Packages

In [ ]:
import matplotlib.pyplot as plt
import lifelines
from lifelines.plotting import plot_lifetimes
import pandas as pd
from scipy import stats

## Task 1

Load the dataset `leukemia`. The DataFrame `leukemia` contains 42 observations. The `t` column is the duration, the `status` column denotes if the event occurred, and the other columns represent variables we wish to regress against:
- `sex`
- `logWBC`: log of white blood cell count
- `Rx`: 0 - new treatment, 1 - placebo


In [ ]:
from lifelines.datasets import load_leukemia
leukemia = load_leukemia()
treatment = (leukemia["Rx"] == 0) # new treatment: treatment = 1

In [ ]:
leukemia.head()

### a) Visualise the data.
- How much censoring is present in each group?
- Are censoring patterns similar between groups?

In [ ]:
plot_lifetimes(leukemia['t'], event_observed=leukemia['status'])
plt.show()

In [ ]:
from lifelines.plotting import plot_lifetimes
plot_lifetimes(leukemia['t'][treatment==0], event_observed=leukemia['status'][treatment==0])
plt.show()

In [ ]:
from lifelines.plotting import plot_lifetimes
plot_lifetimes(leukemia['t'][treatment==1], event_observed=leukemia['status'][treatment==1])
plt.show()

### b) KM estimator and a log-rank test
Estimate the survival functions stratified by treatment group using the Kaplan–Meier estimator. Include 95% confidence intervals and display a risk table showing the number of subjects at risk over time.

First, answer the following questions based only on the Kaplan–Meier curves. Then, verify your conclusions using appropriate functions:
- Are patients receiving the new treatment experiencing better survival compared to those receiving the standard treatment? Use the `logrank_test` to formally assess this.
- What are the median survival times for each group? (Use `kmf.median_survival_time_`)
- What is the probability of surviving beyond 15 months in the new treatment group? (Use `kmf.survival_function_at_times(times=15)`)
- What are the numbers at risk at later time points, and is the remaining sample size sufficient for reliable inference?

In [ ]:
from lifelines.statistics import logrank_test
kmf = lifelines.KaplanMeierFitter(alpha=0.05)

kmf.fit(durations=leukemia['t'][treatment], 
        event_observed=leukemia['status'][treatment], 
        label="Treatment 1")
kmf.plot()

kmf.fit(durations=leukemia['t'][~treatment], 
        event_observed=leukemia['status'][~treatment], 
        label="Treatment 0")
kmf.plot()
plt.show()

In [ ]:
kmf.fit(durations=leukemia['t'][treatment], 
        event_observed=leukemia['status'][treatment], 
        label="Treatment 1")
kmf.plot(at_risk_counts=True)
plt.show()

In [ ]:
kmf.fit(durations=leukemia['t'][~treatment], 
        event_observed=leukemia['status'][~treatment], 
        label="Treatment 1")
kmf.plot(at_risk_counts=True)
plt.show()

### c) KM - continued

The effect of the treatment may differ depending on the level of `logWBC`. CCompare the treatment and placebo groups separately within two subgroups:
- $\textrm{logWBC} < 3$ (low logWBC),
- $\textrm{logWBC} \geq 3$ (high logWBC). 

For each subgroup, compare survival between treatment and placebo using Kaplan–Meier curves and appropriate statistical tests (e.g., the log-rank test). Do you observe evidence of an interaction between treatment effect and `logWBC`?

### d) Cox PH model
Fit three Cox proportional hazards models:
- (1)  with `Rx` only,
- (2)  with `Rx` and `logWBC`
- (3)  with `Rx`, `logWBC` and the interaction term.
Then:
- Compare Model 1 vs Model 2 and Model 2 vs Model 3 using a likelihood ratio test. Which model is sufficient?
- Based on the selected model, Does the new treatment reduce the hazard? hHow does the hazard depend on `logWBC` after adjusting for treatment?
- Using the smaller model (Model 1), plot estimated survival functions for:
  - a patient in the treatment group,
  - a patient in the control group.
- Estimate median survival time from the smaller model for:
  - Treatment group
  - Control group
- Plot estimated survival functions and estimate median survival time from Model 2 for both groups for 5 selected values of `logWBC`.
- Test the following hypotheses in all three models.  How would you explain the conclusion in Model 3?
$$H_0: \beta_{Rx}=0$$
$$H_1: \beta_{Rx}\neq 0$$



In [ ]:
from lifelines import CoxPHFitter
cph0 = CoxPHFitter()
cph0.fit(leukemia, duration_col='t', event_col='status', formula='Rx')

cph1 = CoxPHFitter()
cph1.fit(leukemia, duration_col='t', event_col='status', formula='Rx + logWBC')

cph2 = CoxPHFitter()
cph2.fit(leukemia, duration_col='t', event_col='status', formula='Rx + logWBC + Rx*logWBC')

In [ ]:
cph2.log_likelihood_, cph1.log_likelihood_


In [ ]:
ll0 = cph0.log_likelihood_
ll1 = cph1.log_likelihood_

LR = 2 * (ll1 - ll0)
df = len(cph1.params_) - len(cph0.params_)

p_value = stats.chi2.sf(LR, df)
p_value

In [ ]:
ll1 = cph1.log_likelihood_
ll2 = cph2.log_likelihood_

LR = 2 * (ll2 - ll1)
df = len(cph2.params_) - len(cph1.params_)

p_value = stats.chi2.sf(LR, df)
p_value

In [ ]:
cph1.print_summary()

In [ ]:
cph1.plot()
plt.show()

In [ ]:
cph0.predict_median(pd.DataFrame({'Rx': [0]})), cph0.predict_median(pd.DataFrame({'Rx': [1]}))

In [ ]:
cph0.plot_covariate_groups(covariates='Rx', values=[0,1])
plt.plot()

## Task 2

The task is to investigate the time until re-arrest of individuals released from prison, using survival analysis methods. In particular, analyze whether participation in a financial aid program (`fin`) affects survival time, and how other covariates (e.g., age) contribute to the risk of re-arrest. An individual is a former prisoner released from custody. The event of interest is re-arrest after release, while time zero corresponds to the time of release.

The Rossi dataset (from the study of recidivism) contains follow-up information on individuals released from prison and tracks whether and when they are re-arrested.

In [ ]:
from lifelines.datasets import load_rossi

data = load_rossi()
data.head()

## Task 3

Download the dataset from UCSC Xena:
- [Survival](https://xenabrowser.net/datapages/?dataset=survival%2FLUSC_survival.txt&host=https%3A%2F%2Ftcga.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443)
- [RNAseq](https://xenabrowser.net/datapages/?dataset=TCGA.LUSC.sampleMap%2FHiSeqV2_PANCAN&host=https%3A%2F%2Ftcga.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443).

The goal of this task is to identify genes or pathways that significantly influence overall survival in LUSC (Lung Squamous Cell Carcinoma), measured by survival time (`OS.time`) and censoring indicator (`OS`), using the Cox proportional hazards model. You may explore different approaches to handle the high dimensionality of RNA-seq data, including:
- feature selection prior to modeling (e.g., filtering low-variance genes or using univariate Cox models or log-rank tests),
- dimensionality reduction techniques (e.g. PCA),
- Cox model with regularization (e.g. Lasso).

Identify genes or features associated with increased or decreased hazard. Discuss biological plausibility (if known).

In [ ]:
expr = pd.read_csv("TCGA.LUSC.sampleMap_HiSeqV2_PANCAN.gz", sep="\t", compression="gzip")

In [ ]:
expr.head()

In [ ]:
surv = pd.read_csv("survival_LUSC_survival.txt", sep="\t")

In [ ]:
surv.head()

In [ ]:
surv['sample'] = surv['xena_sample']

expr = expr.set_index("sample")
expr = expr.T
expr['sample'] = expr.index

In [ ]:
df = expr.merge(surv)

In [ ]:
df.head()

In [ ]:
# selection of 100 first genes (not good)
df_cox = df[df.columns[:100].tolist()+['OS', 'OS.time']]
df_cox = df_cox.dropna()

In [ ]:
cph = CoxPHFitter()
cph.fit(df_cox, duration_col='OS', event_col='OS.time')

In [ ]:
cph.print_summary()